# V025 pruning workflow proof

Tests use validation-only real-history replay and explicitly synthetic model fixtures. Production calibration and the tiny forced-branch fixture are separate. Verify scale invariance, warmup/patience/LR grace, unknown-context fallback, source policy hashes, checkpoint-before-stop, exact cloned resumption, queue advancement and exclusion of pruned members from completed candidates. No approved CSV or competition model is fitted.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
import json,hashlib,copy,math,gc
import numpy as np
import torch

def run_pruning_preflight_v025(project_root,output_directory,device='cpu'):
    P=Path(project_root).resolve();O=Path(output_directory);O.mkdir(parents=True,exist_ok=False)
    checks={}
    def check(k,v):
        if not bool(v):raise AssertionError(k)
        checks[k]=True
    def code(p):return '\n'.join(''.join(c['source']) for c in json.loads(Path(p).read_text())['cells'] if c['cell_type']=='code')
    sources={};ns={'__name__':'ventilator_v025_preflight'}
    for name in ['feature_components_v005.ipynb','sequence_components_v005.ipynb','training_components_v008.ipynb','pruning_components_v025.ipynb','training_workflow_v025.ipynb']:
        text=code(P/'research'/name);sources[name]=hashlib.sha256(text.encode()).hexdigest();exec(compile(text,name,'exec'),ns)
    policy_path=P/'research/pruning_policy_v025.json';policy_sha=hashlib.sha256(policy_path.read_bytes()).hexdigest()
    policy=ns['load_pruning_policy_v025'](P,policy_path,policy_sha)
    variant='conv3';known=policy['known_variants'][variant];ref=policy['references'][1]
    descriptor=dict(cohort=copy.deepcopy(policy['cohort']),variant=variant,run_id='synthetic_flat_lagging_curve',
        factory_identity=dict(code_sha256=known['allowed_factory_code_sha256'][0]),model_source_hashes=known['model_source_hashes'])
    budget=policy['cohort']['budget_epochs'];half=policy['start_comparing_epoch']
    bad=2*max(r['best_validation_inspiratory_MAE'][half-1] for r in policy['references'])
    def flat_history(n):
        return [dict(epoch=j+1,cumulative_optimizer_steps=ref['optimizer_steps'][j],best_val_mae=bad,
            validation_inspiratory_mae=bad,lr_used=ref['LR'][j]) for j in range(n)]
    stops=[]
    for e in range(1,budget):
        decision=ns['evaluate_pruning_v025'](flat_history(e),descriptor,policy)
        if decision['action']=='prune':stops.append((e,decision));break
    check('clearly_flat_lagging_curve_can_be_pruned',bool(stops))
    epoch,decision=stops[0]
    check('half_budget_and_full_patience_respected',epoch>=half+policy['patience_intervals'])
    h=flat_history(epoch);perturbed=copy.deepcopy(h)
    for r in perturbed:r['training_eval_inspiratory_mae']=-1e99;r['leaderboard_MAE']=0.
    check('training_MAE_and_leaderboard_fields_do_not_change_decision',
        ns['evaluate_pruning_v025'](h,descriptor,policy)==ns['evaluate_pruning_v025'](perturbed,descriptor,policy))
    unseen=copy.deepcopy(descriptor);unseen['variant']='new_architecture'
    check('unfamiliar_architecture_finishes_representative',ns['evaluate_pruning_v025'](h,unseen,policy)['action']=='continue')
    altered=copy.deepcopy(descriptor);altered['cohort']['fit_breaths']-=1
    check('different_training_size_not_pruned',ns['evaluate_pruning_v025'](h,altered,policy)['action']=='continue')
    altered=copy.deepcopy(descriptor);altered['cohort']['train_cfg']['optimizer']='AdamW'
    check('different_optimizer_not_pruned',ns['evaluate_pruning_v025'](h,altered,policy)['action']=='continue')
    altered=copy.deepcopy(descriptor);altered['cohort']['planned_LR_schedule'][0]*=.5
    check('unfamiliar_LR_schedule_not_pruned',ns['evaluate_pruning_v025'](h,altered,policy)['action']=='continue')
    broken=copy.deepcopy(h);broken[-1]['cumulative_optimizer_steps']+=1
    check('optimizer_step_mismatch_not_pruned',ns['evaluate_pruning_v025'](broken,descriptor,policy)['action']=='continue')
    reductions=ns['major_LR_reductions_v025'](policy['cohort']['planned_LR_schedule'])
    grace_epoch=next(e+1 for e in reductions if e>half+policy['patience_intervals'])
    check('LR_reduction_gets_grace',ns['evaluate_pruning_v025'](flat_history(grace_epoch),descriptor,policy)['reason']=='LR_reduction_grace')
    improving=flat_history(epoch)
    for j,r in enumerate(improving):r['best_val_mae']=bad*(1-.002*(j+1))
    d=ns['evaluate_pruning_v025'](improving,descriptor,policy)
    check('bad_but_still_improving_curve_kept',d['action']=='continue' and not d['validation_improvement_stalled'])
    future=copy.deepcopy(policy)
    for r in future['references']:
        r['best_validation_inspiratory_MAE'][epoch:]=[x*100 for x in r['best_validation_inspiratory_MAE'][epoch:]]
    check('reference_future_epochs_not_used_online',ns['evaluate_pruning_v025'](h,descriptor,future)==decision)
    scaled=copy.deepcopy(policy);scaled_h=copy.deepcopy(h)
    for r in scaled['references']:
        r['best_validation_inspiratory_MAE']=[x*37 for x in r['best_validation_inspiratory_MAE']]
        r['validation_inspiratory_MAE']=[x*37 for x in r['validation_inspiratory_MAE']]
    for r in scaled_h:r['best_val_mae']*=37;r['validation_inspiratory_mae']*=37
    check('relative_rule_is_unit_scale_invariant',ns['evaluate_pruning_v025'](scaled_h,descriptor,scaled)['action']==decision['action'])
    check('completed_budget_never_relabeled_pruned',ns['evaluate_pruning_v025'](flat_history(budget),descriptor,policy)['reason']=='training_budget_complete')
    reviewed=json.loads((P/'research/pruning_calibration_review_v025.json').read_text())
    check('actual_completed_curve_replays_keep_all_three',reviewed['policy_sha256']==policy_sha and reviewed['retrospectively_pruned_runs']==0)
    verified,why=ns['verify_prunable_checkpoint_v025'](O/'missing.pt',dict(epoch=1),8)
    check('missing_checkpoint_cannot_authorize_pruning',verified is None)
    # Tiny, explicitly synthetic training fixture exercises checkpoint/resume/queue mechanics.
    rng=np.random.default_rng(2501);names=['R','C','u_in','u_out','time_step','step','delta_u_in']
    raw=rng.normal(size=(8,80,len(names))).astype(np.float32);raw[:,:,0]=5;raw[:,:,1]=10
    raw[:,:,3]=0;raw[:,45:,3]=1;raw[:,:,4]=np.arange(80)*.03;raw[:,:,5]=np.arange(80)
    target=(10+.05*np.cumsum(raw[:,:,2],axis=1)).astype(np.float32);phase=raw[:,:,3].astype(np.int8)
    spec=dict(family='gru',hidden=4,layers=2,dropout=.15)
    cfg=dict(lr=.001,batch_size=3,max_epochs=8,patience=0,scheduler='constant',prediction_batch_size=3,loss='l1')
    ns['seed_sequence'](2501)
    cohort=ns['pruning_cohort_v025'](raw[:6],target[:6],phase[:6],raw[6:],target[6:],phase[6:],spec,cfg,names,device,'curve_l1')
    fixture_ref=float(np.std(target[6:]))*1e-3
    steps=[2*(j+1) for j in range(8)];rates=[.001]*8
    fixture=dict(schema='v025-validation-only-pruning-policy-v1',origin='synthetic_preflight_only',enabled=True,
        component_code_sha256=sources['pruning_components_v025.ipynb'],reference_file_hashes={},cohort=cohort,
        known_variants={'known_toy':dict(allowed_factory_code_sha256=[None],
            model_source_hashes={'sequence_components_v005.ipynb':sources['sequence_components_v005.ipynb']})},
        references=[dict(id='fixture_ref_'+str(j),optimizer_steps=steps,LR=rates,
            best_validation_inspiratory_MAE=[fixture_ref*(1+j*.1)]*8) for j in range(3)],
        minimum_comparable_references=2,start_comparing_epoch=4,patience_intervals=2,LR_cooldown_evaluations=0,
        relative_deficit_margin=.1,stall_relative_tolerance=1.,
        fixture_note='Forced stopping branch on tiny synthetic data; not the calibrated production policy or a competition score.')
    fixture_path=O/'fixture_policy.json';ns['_workflow_json_v025'](fixture_path,fixture)
    fixture_sha=ns['_workflow_sha_v025'](fixture_path)
    effective=ns['bind_pruning_sources_v025'](P,fixture_path,sources)
    args=(raw[:6],target[:6],phase[:6],raw[6:],target[6:],phase[6:],spec,cfg,2501)
    common=dict(feature_names=names,device=device,loss_kind='curve_l1',arm='v025_proof',source_fingerprints=effective)
    full_report=ns['fit_validate_v008'](*args,O/'full/model.pt',**common)
    stopped=ns['fit_validate_prunable_v025'](*args,O/'pruned/model.pt',project_root=P,policy_path=fixture_path,
        expected_policy_sha256=fixture_sha,variant='known_toy',run_label='known_stop',allow_synthetic_policy=True,**common)
    check('actual_wrapper_returns_pruned_not_completed',stopped['status']=='pruned' and stopped['completed'] is False and stopped['stopped_epoch']<8)
    last=torch.load(stopped['last_checkpoint_file'],map_location='cpu',weights_only=False)
    check('checkpoint_saved_before_stop',last['epoch']==stopped['stopped_epoch'] and not last['completed'] and bool(last['optimizer_state']) and bool(last['rng_state']))
    check('pruned_log_records_reason_progress_and_references',stopped['decision']['reason']=='sustained_same_stage_validation_deficit_and_stalled_improvement' and stopped['decision']['reference_ids'] and stopped['best_epoch']<=stopped['stopped_epoch'])
    ready=ns['candidate_readiness_v025']({'stopped':.5,'complete':.5},{'stopped':stopped,'complete':{'status':'completed'}})
    check('pruned_member_cannot_be_dropped_or_renormalized',not ready['eligible_for_scoring'] and ready['weights']=={'stopped':.5,'complete':.5})
    try:ns['predict_completed_v025'](stopped,raw[6:],device=device)
    except ValueError:checks['pruned_model_rejected_as_completed_candidate']=True
    else:raise AssertionError('Pruned model accepted')
    ns['clone_pruned_state_v025'](stopped,O/'resumed/model.pt')
    resumed=ns['fit_validate_prunable_v025'](*args,O/'resumed/model.pt',project_root=P,policy_path=fixture_path,
        expected_policy_sha256=fixture_sha,variant='known_toy',run_label='resume_audit',allow_synthetic_policy=True,
        pruning_mode='observe_only',mode_reason='Synthetic exact-resume proof; preserve original pruned stage',resume=True,**common)
    full=torch.load(O/'full/model.last.pt',map_location='cpu',weights_only=False)
    res=torch.load(O/'resumed/model.last.pt',map_location='cpu',weights_only=False)
    def equal(a,b):
        if isinstance(a,torch.Tensor):return torch.equal(a,b)
        if isinstance(a,np.ndarray):return np.array_equal(a,b)
        if isinstance(a,dict):return a.keys()==b.keys() and all(equal(v,b[k]) for k,v in a.items())
        if isinstance(a,(tuple,list)):return len(a)==len(b) and all(equal(x,y) for x,y in zip(a,b))
        return a==b
    for key in ['state_dict','optimizer_state','rng_state']:
        check('resume_exact_'+key,equal(full[key],res[key]))
    check('resume_best_native_validation_metric_exact',resumed['status']=='completed' and resumed['report']['validation_mae']==full_report['validation_mae'])
    check('original_pruned_checkpoint_preserved',ns['_workflow_sha_v025'](stopped['last_checkpoint_file'])==stopped['last_checkpoint_sha256'])
    jobs=[dict(id='known_stop',variant='known_toy',fit_args=args+(O/'pruned/model.pt',),fit_kwargs=common),
        dict(id='next_planned',variant='unfamiliar_toy_variant',fit_args=args+(O/'next/model.pt',),
            fit_kwargs=dict(common,arm='v025_next_proof'))]
    queue=ns['run_training_queue_v025'](jobs,project_root=P,policy_path=fixture_path,
        expected_policy_sha256=fixture_sha,state_path=O/'queue.json',allow_synthetic_policy=True)
    check('queue_advances_from_pruned_to_next_experiment',[r['status'] for r in queue]==['pruned','completed'])
    check('unfamiliar_representative_runs_full_budget',queue[1]['report']['executed_epochs']==8)
    # Cached results must retain the same native V008 arm and diagnostic configuration.
    # These calls must reject before fitting; the completed original files stay intact.
    cached_sha=ns['_workflow_sha_v025'](O/'next/model.outcome.json')
    for check_name,override in [
            ('cached_changed_arm_rejected',dict(arm='v025_changed_arm')),
            ('cached_changed_diagnostic_size_rejected',dict(train_diagnostic_breaths=1))]:
        cached_options=dict(common,arm='v025_next_proof');cached_options.update(override)
        try:
            ns['fit_validate_prunable_v025'](*args,O/'next/model.pt',project_root=P,policy_path=fixture_path,
                expected_policy_sha256=fixture_sha,variant='unfamiliar_toy_variant',run_label='next_planned',
                allow_synthetic_policy=True,**cached_options)
        except ValueError as exc:
            check(check_name,'Existing experiment identity changed' in str(exc)
                and ns['_workflow_sha_v025'](O/'next/model.outcome.json')==cached_sha)
        else:raise AssertionError('Changed cached identity accepted: '+check_name)
    pred=ns['predict_completed_v025'](queue[1],raw[6:],device=device,batch_size=3)
    check('completed_native_inference_valid',pred.shape==(2,80) and np.isfinite(pred).all())
    try:ns['load_pruning_policy_v025'](P,fixture_path,'0'*64,True)
    except ValueError:checks['changed_policy_hash_rejected']=True
    else:raise AssertionError('Changed policy accepted')
    try:ns['load_pruning_policy_v025'](P,fixture_path,fixture_sha)
    except ValueError:checks['synthetic_policy_rejected_in_production_mode']=True
    else:raise AssertionError('Synthetic fixture accepted as production')
    result=dict(status='passed',created_utc=datetime.now(timezone.utc).isoformat(),checks=checks,check_count=len(checks),
        source_code_hashes=sources,production_policy_sha256=policy_sha,actual_history_replays=reviewed['leave_target_out_replays'],
        simulated_flat_curve_pruned_epoch=epoch,simulated_only_potential_saved_epochs=budget-epoch,
        fixture_stopped_epoch=stopped['stopped_epoch'],fixture_budget=8,
        exact_pruned_resume=True,next_planned_experiment_executed=True,raw_approved_csv_reads=0,approved_data_fits=0,
        only_synthetic_model_fits=True,actual_cuda=device=='cuda',
        runtime=dict(torch=str(torch.__version__),cuda=torch.version.cuda,numpy=np.__version__,device=device))
    ns['_workflow_json_v025'](O/'proof.json',result)
    print(json.dumps(result,indent=2))
    return result

print('Pruning preflight is inert until explicitly called; tiny model fits are synthetic only.')


Pruning preflight is inert until explicitly called; tiny model fits are synthetic only.
{
  "status": "passed",
  "created_utc": "2026-11-08T05:25:36.578041+00:00",
  "checks": {
    "clearly_flat_lagging_curve_can_be_pruned": true,
    "half_budget_and_full_patience_respected": true,
    "training_MAE_and_leaderboard_fields_do_not_change_decision": true,
    "unfamiliar_architecture_finishes_representative": true,
    "different_training_size_not_pruned": true,
    "different_optimizer_not_pruned": true,
    "unfamiliar_LR_schedule_not_pruned": true,
    "optimizer_step_mismatch_not_pruned": true,
    "LR_reduction_gets_grace": true,
    "bad_but_still_improving_curve_kept": true,
    "reference_future_epochs_not_used_online": true,
    "relative_rule_is_unit_scale_invariant": true,
    "completed_budget_never_relabeled_pruned": true,
    "actual_completed_curve_replays_keep_all_three": true,
    "missing_checkpoint_cannot_authorize_pruning": true,
    "actual_wrapper_returns_pruned_n